# Preprocessing Raw Data

In [53]:
import pandas as pd
import os
import io
import numpy as np
import math
import sys
from pathlib import Path
from scipy.optimize import curve_fit
pd.reset_option("display.precision")
pd.reset_option("display.float_format")
pd.set_option("display.max_columns", None)

### Changing to grandparent directory

In [54]:
# Get the absolute path of the current file
current_file_path = Path("").resolve()
grandparent_dir = current_file_path.parent.parent

# Append this directory to the system path
sys.path.append(str(grandparent_dir))

In [55]:
folder_path = os.path.join(grandparent_dir,"Raw Data - FeCN platinum 8")

## 1. Raw File Import

Importing experimental .dta files in `Raw Data - FeCN platinum 8` folder for all potential modulations.

In [56]:
#Initalizations
dfs = {}
keep_cols = ['Freq','Vmod','IHr1','IHi1', 'VHr1', 
 'VHi1', 'IHr2', 'IHi2', 'VHr2', 'VHi2', 
 'IHr3', 'IHi3','VHr3', 'VHi3', 'IHr4', 
 'IHi4', 'VHr4', 'VHi4', 'IHr5', 'IHi5', 
 'VHr5','VHi5', 'IHr6', 'IHi6', 'VHr6', 
 'VHi6', 'IHr7', 'IHi7', 'VHr7', 'VHi7',
 'IHr8', 'IHi8', 'VHr8', 'VHi8', 'IHr9', 
 'IHi9', 'VHr9', 'VHi9', 'IHr10','IHi10', 
 'VHr10', 'VHi10']

# Iterate over EISPOT files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith('.DTA') and filename.startswith("EISPOT"):
        file_path = os.path.join(folder_path, filename)
        
        # Open and read the file content, skipping instrument settings
        with open(file_path, "r", encoding="ISO-8859-1") as f:
            content = f.readlines()
            for i,line in enumerate(content):
                if line.__contains__("ZCURVE"):
                    index = i + 1 #finds line where datatable resides (below "Zcurve" label)
                    break
            content = content[index:]
                
            # Read the remaining lines and create a DataFrame
            df = pd.DataFrame([line.split('\t') for line in content if line.strip()])
            df = df.map(lambda x: x.strip() if isinstance(x, str) else x) # Remove newline characters from the last column
            df = df.drop(0, axis=1) #drops first column
            df.columns = df.iloc[0] # Sets the header names to values from the first row
            df = df.drop(0) # Drop the first row (since it's now used as header)
            df = df.iloc[1:] # Drops row with units
            df = df.dropna(how='all')
            df = df.reset_index(drop=True)
            df = df[keep_cols]
            
            #Saves Files to Output Folder
            filename_stripped = filename[:-4]
            dfs[filename_stripped] = df


In [57]:
#Generating the Potnetial Modulation, File Numbers, and Rank Row Labels
keys = dfs.keys()
dfs_list = []
for key in keys:
    names = key.split("_")
    modulation = names[-2]
    file_num = names[-1]
    df_temp = dfs[key]
    df_temp["Modulation"] = modulation
    df_temp["File Number"] = file_num
    df_temp["Rank"] = pd.Series(range(len(df_temp), 0, -1)) #ranks each frequency row in increasing order from 1 to 73 
    dfs_list.append(df_temp)
df_all = pd.concat(dfs_list,axis=0) #concatenates all potential modulations with the generated row labels
df_all.head()


,Freq,Vmod,IHr1,IHi1,VHr1,VHi1,IHr2,IHi2,VHr2,VHi2,IHr3,IHi3,VHr3,VHi3,IHr4,IHi4,VHr4,VHi4,IHr5,IHi5,VHr5,VHi5,IHr6,IHi6,VHr6,VHi6,IHr7,IHi7,VHr7,VHi7,IHr8,IHi8,VHr8,VHi8,IHr9,IHi9,VHr9,VHi9,IHr10,IHi10,VHr10,VHi10,Modulation,File Number,Rank
0,1500059,0.0100416,-0.1554099,0.1005158,-0.5449082,0.7273971,0.0001771,0.0003456,0.0022918,0.0003195,0.0001909,2.13747E-005,0.0004798,-0.0005833,-1.733797E-005,-1.892305E-005,-0.000109,-0.0001465,-2.630125E-005,-2.461887E-005,3.031921E-005,-8.406351E-005,-1.053838E-005,-3.035442E-005,-0.0001021,-0.0003127,-2.854248E-005,-3.008579E-005,9.064237E-005,-7.600245E-005,-1.538056E-005,1.34838E-005,-8.371008E-005,-0.0001187,4.489208E-005,-2.873351E-005,2.263952E-005,-6.906456E-005,3.511109E-005,2.369168E-005,5.62612E-005,0.0001209,10mV,2,73
1,1191504,0.0127297,-0.1884086,-0.1575368,-1.05014,-0.4740141,-0.0005022,-0.0004305,-0.0034932,-0.0001822,-0.0001355,-9.377198E-005,-0.0005188,0.0002364,-3.561616E-005,-2.267352E-007,-0.0001384,-1.318532E-005,-2.174279E-005,-2.626035E-005,-0.0003523,0.0001501,-2.981224E-005,1.589712E-005,-0.0001955,6.773416E-005,9.188901E-006,-3.798175E-005,-0.0002933,-4.469837E-005,5.080743E-006,1.137075E-005,-0.0001775,3.111432E-005,9.50952E-006,3.339876E-006,7.893879E-005,0.0002979,-3.545116E-005,6.209761E-006,-0.000311,4.340522E-005,10mV,2,72
2,946464.9,0.0125644,-0.2247906,-0.1077484,-1.107873,-0.2566108,-0.0004818,-0.0002192,-0.0025545,0.0002215,-0.0001282,0.0001399,-0.0003222,0.0005156,1.005217E-005,-3.252645E-005,0.0001972,-0.000221,-3.008847E-005,1.714582E-005,-0.0002009,0.000115,-9.3875E-006,2.866518E-005,0.0001298,-3.686641E-005,-2.070225E-005,3.939122E-005,-0.0000951,0.0002931,-9.987737E-006,7.646624E-006,-0.0001524,5.3528E-005,-3.220413E-006,8.119736E-006,-0.000218,0.000129,-3.121723E-005,-2.413802E-005,-0.0003514,1.44979E-005,10mV,2,71
3,751816.4,0.0125862,-0.2421683,-0.0776115,-1.130004,-0.1442501,-0.000378,-8.300774E-005,-0.0017554,0.0001786,-6.02461E-005,8.408003E-005,-0.0001097,-4.614006E-005,2.129271E-005,2.460842E-005,0.0001312,0.0001698,1.643202E-006,-1.570839E-005,-8.88249E-005,7.085875E-005,-1.629349E-005,-3.153985E-005,-0.0001818,-0.0001072,-1.555102E-005,-2.120703E-005,-0.0001035,-4.086062E-005,-4.300034E-006,2.475572E-006,0.0001769,-0.0001566,7.459836E-006,3.618654E-005,0.0001038,0.0001558,-2.017478E-007,-5.072624E-006,0.0000988,0.0002441,10mV,2,70
4,597246.1,0.012618,-0.1980543,-0.1649427,-0.981439,-0.5840084,-0.0001852,-0.0002086,-0.0008408,-0.0006683,-9.225496E-005,-0.0001134,-0.000529,-0.0005786,2.091425E-005,-3.783447E-005,-0.0001304,-0.0002228,-2.214755E-005,2.339273E-005,-4.814566E-005,-0.0001464,1.862971E-005,2.310216E-005,1.890911E-005,0.0001798,-1.425622E-006,-1.560117E-005,1.21491E-005,7.170206E-005,-1.143944E-005,-1.896254E-005,0.0001144,-4.286413E-005,-1.001463E-005,2.083322E-005,-0.0002315,5.342066E-005,4.235655E-006,-8.113683E-006,0.0001194,0.0001447,10mV,2,69


#### The units of `Frequency` (Freq) is Hz, `Voltage` (Vmod, VHi, VHr) is Volts, and `Current` (IHi, IHr) is Amps, unless otherwise specified in the column name.

# 2. Modulation Factor

Correction factor applied by the Gamry Instrument is re-corrected.

In [58]:
N = 128
mod = np.sqrt(2)/N #correction factor re-correction
mod_cols = ["IHr","IHi","VHr","VHi"]
nums = range(1, 11)
for num in nums:
    for col in mod_cols:
        cname = f"{col}{num}"
        df_all[cname] = df_all[cname].astype(float) * mod #multiplies each current and voltage by mod  
save_path = os.path.join(current_file_path,"unprocessed_data_modFactor.csv")
df_all.to_csv(save_path,index=False)
df_all.head()

,Freq,Vmod,IHr1,IHi1,VHr1,VHi1,IHr2,IHi2,VHr2,VHi2,IHr3,IHi3,VHr3,VHi3,IHr4,IHi4,VHr4,VHi4,IHr5,IHi5,VHr5,VHi5,IHr6,IHi6,VHr6,VHi6,IHr7,IHi7,VHr7,VHi7,IHr8,IHi8,VHr8,VHi8,IHr9,IHi9,VHr9,VHi9,IHr10,IHi10,VHr10,VHi10,Modulation,File Number,Rank
0,1500059,0.0100416,-0.001717,0.001111,-0.006020,0.008037,0.000002,3.818377e-06,0.000025,0.000004,2.109167e-06,2.361593e-07,0.000005,-6.444615e-06,-1.915593e-07,-2.090721e-07,-0.000001,-1.618612e-06,-2.905905e-07,-2.720027e-07,3.349831e-07,-9.287793e-07,-1.164337e-07,-3.353721e-07,-1.128056e-06,-3.454880e-06,-3.153528e-07,-3.324042e-07,1.001466e-06,-8.397164e-07,-1.699328e-07,1.489764e-07,-9.248745e-07,-1.311462e-06,4.959921e-07,-3.174634e-07,2.501337e-07,-7.630628e-07,3.879264e-07,2.617586e-07,6.216043e-07,1.335769e-06,10mV,2,73
1,1191504,0.0127297,-0.002082,-0.001741,-0.011603,-0.005237,-0.000006,-4.756398e-06,-0.000039,-0.000002,-1.497078e-06,-1.036044e-06,-0.000006,2.611876e-06,-3.935067e-07,-2.505094e-09,-0.000002,-1.456786e-07,-2.402262e-07,-2.901386e-07,-3.892402e-06,1.658386e-06,-3.293818e-07,1.756400e-07,-2.159990e-06,7.483638e-07,1.015240e-07,-4.196430e-07,-3.240538e-06,-4.938519e-07,5.613481e-08,1.256302e-07,-1.961116e-06,3.437679e-07,1.050663e-07,3.690077e-08,8.721587e-07,3.291361e-06,-3.916837e-07,6.860881e-08,-3.436097e-06,4.795645e-07,10mV,2,72
2,946464.9,0.0125644,-0.002484,-0.001190,-0.012240,-0.002835,-0.000005,-2.421841e-06,-0.000028,0.000002,-1.416423e-06,1.545691e-06,-0.000004,5.696629e-06,1.110618e-07,-3.593699e-07,0.000002,-2.441728e-06,-3.324338e-07,1.894363e-07,-2.219652e-06,1.270582e-06,-1.037182e-07,3.167085e-07,1.434101e-06,-4.073201e-07,-2.287297e-07,4.352156e-07,-1.050716e-06,3.238328e-06,-1.103499e-07,8.448406e-08,-1.683798e-06,5.914064e-07,-3.558087e-08,8.971126e-08,-2.408582e-06,1.425262e-06,-3.449049e-07,-2.666900e-07,-3.882458e-06,1.601807e-07,10mV,2,71
3,751816.4,0.0125862,-0.002676,-0.000857,-0.012485,-0.001594,-0.000004,-9.171146e-07,-0.000019,0.000002,-6.656317e-07,9.289619e-07,-0.000001,-5.097805e-07,2.352534e-07,2.718872e-07,0.000001,1.876043e-06,1.815499e-08,-1.735548e-07,-9.813858e-07,7.828860e-07,-1.800193e-07,-3.484694e-07,-2.008625e-06,-1.184404e-06,-1.718161e-07,-2.343068e-07,-1.143524e-06,-4.514503e-07,-4.750911e-08,2.735146e-08,1.954487e-06,-1.730202e-06,8.242032e-08,3.998086e-07,1.146839e-06,1.721363e-06,-2.229019e-09,-5.604511e-08,1.091596e-06,2.696949e-06,10mV,2,70
4,597246.1,0.012618,-0.002188,-0.001822,-0.010843,-0.006452,-0.000002,-2.304726e-06,-0.000009,-0.000007,-1.019283e-06,-1.252905e-06,-0.000006,-6.392687e-06,2.310720e-07,-4.180158e-07,-0.000001,-2.461615e-06,-2.446982e-07,2.584556e-07,-5.319394e-07,-1.617507e-06,2.058312e-07,2.552452e-07,2.089181e-07,1.986528e-06,-1.575105e-08,-1.723702e-07,1.342299e-07,7.922033e-07,-1.263891e-07,-2.095084e-07,1.263953e-06,-4.735862e-07,-1.106471e-07,2.301767e-07,-2.557738e-06,5.902205e-07,4.679782e-08,-8.964438e-08,1.319196e-06,1.598724e-06,10mV,2,69


# 3. Rotation

Rotating I1 and I2 vectors by the angle of rotation found to make VHi1 ≈ 0.

In [59]:
df_all["theta"] = -np.arctan2(df_all["VHi1"],df_all["VHr1"]) #arctan2 returns a theta between -180 and 180 degrees, representing the angle from the positive x-axis
real_cols = ["IHr","VHr"]
imag_cols = ["IHi","VHi"]
nums = range(1, 11)
for num in nums:
    for real,imag in zip(real_cols,imag_cols):
        rname = f"{real}{num}"
        iname = f"{imag}{num}"
        r_orig = df_all[rname].astype(float).copy()
        i_orig = df_all[iname].astype(float).copy()
        df_all[rname] = np.cos(num * df_all["theta"]) * r_orig - np.sin(num * df_all["theta"]) * i_orig
        df_all[iname] = np.sin(num * df_all["theta"]) * r_orig + np.cos(num * df_all["theta"]) * i_orig
A1 = (df_all["IHr1"] + df_all["IHi1"]*1j) / (df_all["VHr1"] + df_all["VHi1"]*1j)
A2 = (df_all["IHr2"] + df_all["IHi2"]*1j) / (df_all["VHr1"] + df_all["VHi1"]*1j)**2
df_all["Re{I1} Rot"] = df_all["IHr1"]
df_all["Im{I1} Rot"] = df_all["IHi1"]
df_all["Re{I2} Rot"] = df_all["IHr2"]
df_all["Im{I2} Rot"] = df_all["IHi2"]
df_all["Re{A1} Rot"] = A1.apply(lambda x: x.real)
df_all["Im{A1} Rot"] = A1.apply(lambda x: x.imag)
df_all["Re{A2} Rot"] = A2.apply(lambda x: x.real)
df_all["Im{A2} Rot"] = A2.apply(lambda x: x.imag)
save_path = os.path.join(current_file_path,"rotated_data_noCdl.csv")
df_all.to_csv(save_path,index=False)
df_all


,Freq,Vmod,IHr1,IHi1,VHr1,VHi1,IHr2,IHi2,VHr2,VHi2,IHr3,IHi3,VHr3,VHi3,IHr4,IHi4,VHr4,VHi4,IHr5,IHi5,VHr5,VHi5,IHr6,IHi6,VHr6,VHi6,IHr7,IHi7,VHr7,VHi7,IHr8,IHi8,VHr8,VHi8,IHr9,IHi9,VHr9,VHi9,IHr10,IHi10,VHr10,VHi10,Modulation,File Number,Rank,theta,Re{I1} Rot,Im{I1} Rot,Re{I2} Rot,Im{I2} Rot,Re{A1} Rot,Im{A1} Rot,Re{A2} Rot,Im{A2} Rot
0,1500059,0.0100416,0.001918,0.000708,0.010042,-1.734723e-18,-4.214425e-06,8.045387e-07,-1.050496e-05,2.330799e-05,2.058186e-06,-5.179061e-07,2.706653e-06,-7.893584e-06,4.849657e-08,2.793818e-07,1.407607e-07,2.012562e-06,2.500537e-07,-3.096796e-07,9.507638e-07,2.662572e-07,-3.079772e-07,-1.765823e-07,-3.118802e-06,-1.865953e-06,2.384193e-07,3.912751e-07,-1.155626e-06,6.103982e-07,-2.063573e-07,-9.212900e-08,8.049724e-07,-1.388289e-06,-4.272261e-08,-5.873380e-07,-5.516340e-07,-5.835509e-07,-4.219200e-07,-2.024549e-07,-8.095620e-07,-1.230967e-06,10mV,2,73,-2.213734,0.001918,0.000708,-4.214425e-06,8.045387e-07,0.191033,0.070546,-0.041796,0.007979
1,1191504,0.0127297,0.002613,0.000730,0.012730,1.734723e-18,-7.237396e-06,1.014959e-06,-2.703940e-05,2.761303e-05,1.430825e-06,-1.125775e-06,-8.088686e-07,-6.246861e-06,4.665821e-08,3.907388e-07,4.642824e-08,1.535340e-06,1.220666e-07,-3.563548e-07,-3.446401e-06,-2.454254e-06,3.711226e-07,4.012225e-08,2.206726e-06,5.966279e-07,1.724730e-07,-3.958036e-07,-3.106553e-06,-1.046089e-06,-8.551716e-08,-1.078002e-07,1.814751e-06,-8.190439e-07,1.051062e-07,-3.678703e-08,2.736507e-06,2.026141e-06,1.171176e-07,-3.800089e-07,1.136351e-06,-3.278026e-06,10mV,2,72,2.717590,0.002613,0.000730,-7.237396e-06,1.014959e-06,0.205298,0.057347,-0.044663,0.006263
2,946464.9,0.0125644,0.002688,0.000599,0.012564,2.168404e-18,-5.845884e-06,1.651927e-07,-2.427336e-05,1.460682e-05,1.235239e-07,-2.092883e-06,-8.328516e-07,-6.665614e-06,-2.156969e-07,-3.081498e-07,-5.919694e-07,-3.218487e-06,-3.256416e-08,-3.812321e-07,-2.226234e-07,-2.547877e-06,2.889416e-07,1.660556e-07,-1.066612e-07,-1.487003e-06,-4.402471e-07,-2.188889e-07,-3.261128e-06,-9.776584e-07,1.091665e-07,8.600778e-08,9.897349e-07,1.485046e-06,-9.602635e-08,9.646117e-09,-2.373031e-06,-1.483701e-06,2.053230e-08,4.355013e-07,2.638853e-06,2.852296e-06,10mV,2,71,2.913982,0.002688,0.000599,-5.845884e-06,1.651927e-07,0.213951,0.047701,-0.037031,0.001046
3,751816.4,0.0125862,0.002763,0.000512,0.012586,2.168404e-19,-4.272812e-06,1.614589e-07,-1.827694e-05,6.782213e-06,2.725751e-07,-1.109837e-06,1.314674e-06,2.266245e-08,3.377839e-07,1.231620e-07,2.178961e-06,9.343001e-07,8.830852e-08,1.505074e-07,3.258907e-07,-1.212363e-06,-3.707842e-07,-1.278943e-07,-2.270932e-06,5.293904e-07,2.901990e-07,1.431768e-08,1.071297e-06,-6.031398e-07,-1.791541e-09,5.479059e-08,-4.404566e-07,-2.572862e-06,-3.979458e-07,-9.098981e-08,-2.042118e-06,3.287602e-07,-5.418448e-08,-1.449360e-08,2.899356e-06,-2.425963e-07,10mV,2,70,3.014625,0.002763,0.000512,-4.272812e-06,1.614589e-07,0.219498,0.040663,-0.026973,0.001019
4,597246.1,0.012618,0.002812,0.000447,0.012618,1.734723e-18,-3.001671e-06,6.990197e-07,-1.092079e-05,4.642522e-06,1.211648e-06,-1.067997e-06,6.156736e-06,-6.092728e-06,-4.764168e-07,3.403727e-08,-1.278935e-06,2.549427e-06,-3.337223e-07,1.237171e-07,2.376026e-07,-1.686070e-06,-2.253451e-07,-2.381936e-07,-3.651422e-07,-1.963826e-06,-1.124243e-07,-1.316068e-07,5.671745e-07,5.691369e-07,2.427819e-07,-3.041344e-08,-8.053259e-08,1.347359e-06,2.416501e-07,8.263993e-08,8.886590e-07,2.469953e-06,9.959343e-08,-1.752994e-08,-4.635514e-07,2.020227e-06,10mV,2,69,2.604818,0.002812,0.000447,-3.001671e-06,6.990197e-07,0.222885,0.035434,-0.018853,0.004390
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,0.238186,0.0200097,0.000011,0.000010,0.020010,1.626303e-18,-1.635020e-07,-1.475693e-07,-2.797021e-08,2.294070e-07,-4.310682e-07,-3.760629e-07,-7.506153e-08,-4.478089e-08,1.051897e-08,1.205506e-08,

#### The units of `A1` (Re{A1}, Im{A1}) is A/V and `A2` (Re{A2}, Im{A2}) is A/V<sup>2</sup>.

# 4. Cdl Correction (12 Lowest Frequencies $\omega$ ONLY)

Finding the best fit Cdl to shift the I1 vectors upwards to the 45–degree line. This effect was only added to the 12 lowest frequencies.

In [60]:
def Cdl_model(xdata, Cdl):
    f, E, Iimag = xdata
    return Iimag + Cdl * f * math.pi * 2 * E

#Filtering to 12 Lowest Frequencies and Weakly Nonlinear Modulations
mod_used = ["1mV","5mV","10mV","20mV"]
df_12_freq_filter = df_all[(df_all["Rank"] <= 12) & (df_all["Modulation"].isin(mod_used))]

# Fit
xdata = (df_12_freq_filter["Freq"],df_12_freq_filter["VHr1"], df_12_freq_filter["Im{I1} Rot"])
p0 = [1e-5]
popt, pcov = curve_fit(Cdl_model, xdata, df_12_freq_filter["Re{I1} Rot"], p0=p0) #uses a summed squared error for real and imaginary

Cdl_fit = popt[0]
Cdl_err = np.sqrt(np.diag(pcov))[0]
print("Best fit Cdl", Cdl_fit)
print("Standard error:", Cdl_err)

Best fit Cdl 1.2077942870333914e-05
Standard error: 2.4484945209296043e-07


In [61]:
#Correct IHi1 Using the Fit Cdl
df_12_freq = df_all[df_all["Rank"] <= 12].copy()
df_12_freq['IHi1'] = df_12_freq['IHi1'] + \
    Cdl_fit * df_12_freq["Freq"].astype(float) * 2 * math.pi * df_12_freq["VHr1"]  
    
#Add Processed Data Columns
A1 = (df_12_freq["IHr1"] + df_12_freq["IHi1"]*1j) / (df_12_freq["VHr1"] + df_12_freq["VHi1"]*1j)
A2 = (df_12_freq["IHr2"] + df_12_freq["IHi2"]*1j) / (df_12_freq["VHr1"] + df_12_freq["VHi1"]*1j)**2
df_12_freq["Re{A1} Processed"] = A1.apply(lambda x: x.real)
df_12_freq["Im{A1} Processed"] = A1.apply(lambda x: x.imag)
df_12_freq["Re{A2} Processed"] = A2.apply(lambda x: x.real)
df_12_freq["Im{A2} Processed"] = A2.apply(lambda x: x.imag)
df_12_freq["I1 Mag (A * 10^6)"] = np.sqrt(df_12_freq["IHr1"]**2 + df_12_freq["IHi1"]**2) * 10**6
df_12_freq["I2 Mag (A * 10^6)"] = np.sqrt(df_12_freq["IHr2"]**2 + df_12_freq["IHi2"]**2) * 10**6
df_12_freq["I3 Mag (A * 10^6)"] = np.sqrt(df_12_freq["IHr3"]**2 + df_12_freq["IHi3"]**2) * 10**6
df_12_freq["File Number"] = df_12_freq["File Number"].astype(int)
order = ['1mV', '5mV', '10mV', '20mV', '50mV', '100mV']  # your custom order
df_12_freq["Modulation"] = pd.Categorical(
    df_12_freq["Modulation"], 
    categories=order, 
    ordered=True
)
df_12_freq.sort_values(by=["Modulation","File Number","Rank"],ascending=[True,True,True],inplace=True)
df_12_freq.reset_index(inplace=True,drop=True)
save_path = os.path.join(current_file_path,"preprocessed_data_final.csv")
df_12_freq.to_csv(save_path,index=False)
df_12_freq

,Freq,Vmod,IHr1,IHi1,VHr1,VHi1,IHr2,IHi2,VHr2,VHi2,IHr3,IHi3,VHr3,VHi3,IHr4,IHi4,VHr4,VHi4,IHr5,IHi5,VHr5,VHi5,IHr6,IHi6,VHr6,VHi6,IHr7,IHi7,VHr7,VHi7,IHr8,IHi8,VHr8,VHi8,IHr9,IHi9,VHr9,VHi9,IHr10,IHi10,VHr10,VHi10,Modulation,File Number,Rank,theta,Re{I1} Rot,Im{I1} Rot,Re{I2} Rot,Im{I2} Rot,Re{A1} Rot,Im{A1} Rot,Re{A2} Rot,Im{A2} Rot,Re{A1} Processed,Im{A1} Processed,Re{A2} Processed,Im{A2} Processed,I1 Mag (A * 10^6),I2 Mag (A * 10^6),I3 Mag (A * 10^6)
0,0.0946587,0.0009973,3.591199e-07,3.523981e-07,0.000997,-2.202286e-20,8.663635e-11,-8.126279e-10,4.232798e-07,-3.294868e-08,7.626110e-10,2.017523e-10,2.346997e-07,5.762950e-08,-2.373707e-10,2.821689e-10,-1.052656e-08,-1.447066e-07,-6.530927e-10,-3.320308e-10,-3.341308e-07,2.547215e-08,2.656201e-11,-4.088813e-10,-1.537204e-07,8.695272e-08,4.448213e-10,1.132335e-10,1.858028e-07,-1.037471e-07,2.853310e-11,3.017305e-10,1.767531e-07,1.237498e-07,-3.898226e-10,-3.083133e-10,-3.397066e-07,-4.568547e-08,2.740109e-11,-1.087701e-10,5.213099e-08,-1.581060e-08,1mV,1,1,1.577347,3.591199e-07,3.452343e-07,8.663635e-11,-8.126279e-10,0.000360,0.000346,0.000087,-0.000817,0.000360,0.000353,0.000087,-0.000817,0.503142,0.000817,0.000789
1,0.1193355,0.001,4.035293e-07,3.971887e-07,0.001000,-2.032879e-20,-9.339797e-11,-6.487907e-10,4.785573e-07,2.114960e-07,4.467975e-10,2.264564e-10,3.137436e-07,4.377546e-08,-1.865087e-10,1.421533e-10,-1.666648e-07,2.663718e-08,-5.465652e-10,-3.272127e-10,-4.611297e-07,-3.620670e-08,-1.118692e-11,-2.238098e-10,-5.510873e-08,-1.038334e-08,2.791711e-10,2.311489e-10,2.329712e-07,1.051521e-07,7.558033e-11,2.107923e-10,4.213944e-08,-1.177404e-08,-2.730901e-10,-2.024192e-10,-1.532516e-07,-5.505051e-08,9.660679e-11,-1.194197e-10,1.403687e-07,-2.428563e-08,1mV,1,2,1.581362,4.035293e-07,3.881329e-07,-9.339797e-11,-6.487907e-10,0.000404,0.000388,-0.000093,-0.000649,0.000404,0.000397,-0.000093,-0.000649,0.566211,0.000655,0.000501
2,0.1502404,0.0009999,4.523700e-07,4.460860e-07,0.001000,9.486769e-20,-1.935061e-10,-3.516726e-10,2.733893e-07,4.409188e-08,5.004613e-10,4.626549e-10,4.484382e-07,-7.147300e-08,-8.007073e-11,8.420214e-11,9.365113e-08,-1.839536e-08,-5.409837e-10,-3.938363e-10,-3.415950e-07,5.135947e-08,-7.067951e-11,-1.968812e-10,3.564944e-08,-1.417029e-07,2.996963e-10,3.998319e-11,9.160415e-08,-1.148292e-07,4.849020e-11,2.310708e-10,4.221248e-09,1.903346e-08,-2.188515e-10,-1.487758e-10,-1.189714e-07,1.105995e-08,-5.979940e-11,3.879897e-11,-2.500749e-08,1.423364e-07,1mV,1,3,1.589897,4.523700e-07,4.346854e-07,-1.935061e-10,-3.516726e-10,0.000452,0.000435,-0.000194,-0.000352,0.000452,0.000446,-0.000194,-0.000352,0.635320,0.000401,0.000682
3,0.1890121,0.001,5.059514e-07,5.007543e-07,0.001000,-2.032879e-20,6.220624e-11,-5.464256e-10,2.752424e-07,-1.908649e-07,5.522691e-10,3.574026e-10,4.747942e-07,3.431016e-07,-3.802952e-10,-3.650356e-11,-2.333305e-07,3.051192e-08,-5.248978e-10,-3.885814e-10,-2.706785e-07,-8.015905e-08,5.126480e-11,-1.614626e-10,-2.730530e-08,-7.748513e-08,2.767869e-10,2.959406e-10,9.182973e-08,1.126101e-07,1.057279e-10,2.010512e-10,9.683833e-08,-3.234306e-08,-2.536576e-10,-3.125619e-10,-1.943091e-07,-5.522583e-08,8.328654e-11,-1.107785e-10,-6.161893e-08,-4.134967e-08,1mV,1,4,1.601049,5.059514e-07,4.864112e-07,6.220624e-11,-5.464256e-10,0.000506,0.000486,0.000062,-0.000546,0.000506,0.000501,0.000062,-0.000546,0.711858,0.000550,0.000658
4,0.238186,0.0010009,5.671599e-07,5.633204e-07,0.001001,-6.776264e-20,8.054224e-11,-3.527037e-10,4.929767e-07,4.224212e-08,5.100490e-10,2.622421e-10,3.830076e-07,1.658535e-07,-4.290259e-10,-2.392035e-11,-2.023576e-07,-2.439399e-08,-7.498274e-10,-5.066907e-10,-3.159136e-07,-4.297256e-08,-2.356455e-10,-1.611189e-10,-2.334416e-07,2.499535e-07,2.905577e-10,1.147172e-10,1.468195e-07,-5.970987e-08,8.559982e-11,1.009345e-10,1.340114e-07,5.087221e-08,-2.678417e-10,-4.315932e-10,-1.923184e-07,-9.345353e-08,2.021909e-10,-1.494850e-10,8.742409e-08,-5.469920e-08,1mV,1,5,1.615278,5.671599e-07,5.452294e-07,8.054224e-11,-3.527037e-10,0.00

#### These are the final processed IHj and VHj vectors used in the paper. `A1 Rot` represent the 2nd-order admittances prior to adding the capacitive effect, while `A1 Processed` represent the fully processed 2nd-order admittances. `A2 Rot` and `A2 Processed` are the same (since no capacitive effect is added) but were created to alleviate confusion during plotting.

# 5. Addmittance Columns for Parameter Fitting (12 Lowest Frequencies $\omega$ ONLY)

In [62]:
F = 96485
R = 8.314
T = 298
f_val =  F / (R * T)

### Method 1 - Frequency Normalized Current Magnitudes

In [63]:
df_12_freq["Freq"] = df_12_freq["Freq"].astype(float)
df_12_freq["I1_mag_div_g1"] = df_12_freq["I1 Mag (A * 10^6)"] / (np.sqrt(df_12_freq["Freq"] * 2 * math.pi/2) * f_val * np.sqrt(2))
df_12_freq["I2_mag_div_g2"] = df_12_freq["I2 Mag (A * 10^6)"] / (np.sqrt(2 * df_12_freq["Freq"] * 2 * math.pi / 2) * f_val**2 * np.sqrt(2) / 8)

h1_2nd = [1] * len(df_12_freq)
h1_4th = 1 - df_12_freq["VHr1"]**2 * f_val**2 / 16
h1_6th = 1 - df_12_freq["VHr1"]**2 * f_val**2 / 16 + df_12_freq["VHr1"]**4 * f_val**4 / 192

h2_2nd = [1] * len(df_12_freq)
h2_4th = 1 - 7 * df_12_freq["VHr1"]**2 * f_val**2 / 24
h2_6th = 1 - 7 * df_12_freq["VHr1"]**2 * f_val**2 / 24 + 19 * df_12_freq["VHr1"]**4 * f_val**4 / 256

df_12_freq["h1_2nd"] = h1_2nd
df_12_freq["h1_4th"] = h1_4th
df_12_freq["h1_6th"] = h1_6th
df_12_freq["h2_2nd"] = h2_2nd
df_12_freq["h2_4th"] = h2_4th
df_12_freq["h2_6th"] = h2_6th

### Method 2 - Nonlinearly Corrected Admittances

In [64]:
h1s = [h1_2nd,h1_4th,h1_6th]
h2s = [h2_2nd,h2_4th,h2_6th]
orders = ["2nd","4th","6th"]
for h1,h2,order in zip(h1s,h2s,orders):
    A1 = (df_12_freq["IHr1"] + df_12_freq["IHi1"]*1j) / ((df_12_freq["VHr1"] + df_12_freq["VHi1"]*1j) * h1)
    A2 = (df_12_freq["IHr2"] + df_12_freq["IHi2"]*1j) / (((df_12_freq["VHr1"] + df_12_freq["VHi1"]*1j)**2) * h2)
    df_12_freq[f"Re{{A1}}, {order} order"] = A1.apply(lambda x: x.real)
    df_12_freq[f"Im{{A1}}, {order} order"] = A1.apply(lambda x: x.imag)
    df_12_freq[f"Re{{A2}}, {order} order"] = A2.apply(lambda x: x.real)
    df_12_freq[f"Im{{A2}}, {order} order"] = A2.apply(lambda x: x.imag)

## Final Result:

In [65]:
df_12_freq

,Freq,Vmod,IHr1,IHi1,VHr1,VHi1,IHr2,IHi2,VHr2,VHi2,IHr3,IHi3,VHr3,VHi3,IHr4,IHi4,VHr4,VHi4,IHr5,IHi5,VHr5,VHi5,IHr6,IHi6,VHr6,VHi6,IHr7,IHi7,VHr7,VHi7,IHr8,IHi8,VHr8,VHi8,IHr9,IHi9,VHr9,VHi9,IHr10,IHi10,VHr10,VHi10,Modulation,File Number,Rank,theta,Re{I1} Rot,Im{I1} Rot,Re{I2} Rot,Im{I2} Rot,Re{A1} Rot,Im{A1} Rot,Re{A2} Rot,Im{A2} Rot,Re{A1} Processed,Im{A1} Processed,Re{A2} Processed,Im{A2} Processed,I1 Mag (A * 10^6),I2 Mag (A * 10^6),I3 Mag (A * 10^6),I1_mag_div_g1,I2_mag_div_g2,h1_2nd,h1_4th,h1_6th,h2_2nd,h2_4th,h2_6th,"Re{A1}, 2nd order","Im{A1}, 2nd order","Re{A2}, 2nd order","Im{A2}, 2nd order","Re{A1}, 4th order","Im{A1}, 4th order","Re{A2}, 4th order","Im{A2}, 4th order","Re{A1}, 6th order","Im{A1}, 6th order","Re{A2}, 6th order","Im{A2}, 6th order"
0,0.094659,0.0009973,3.591199e-07,3.523981e-07,0.000997,-2.202286e-20,8.663635e-11,-8.126279e-10,4.232798e-07,-3.294868e-08,7.626110e-10,2.017523e-10,2.346997e-07,5.762950e-08,-2.373707e-10,2.821689e-10,-1.052656e-08,-1.447066e-07,-6.530927e-10,-3.320308e-10,-3.341308e-07,2.547215e-08,2.656201e-11,-4.088813e-10,-1.537204e-07,8.695272e-08,4.448213e-10,1.132335e-10,1.858028e-07,-1.037471e-07,2.853310e-11,3.017305e-10,1.767531e-07,1.237498e-07,-3.898226e-10,-3.083133e-10,-3.397066e-07,-4.568547e-08,2.740109e-11,-1.087701e-10,5.213099e-08,-1.581060e-08,1mV,1,1,1.577347,3.591199e-07,3.452343e-07,8.663635e-11,-8.126279e-10,0.000360,0.000346,0.000087,-0.000817,0.000360,0.000353,0.000087,-0.000817,0.503142,0.000817,0.000789,0.016753,0.000004,1,0.999906,0.999906,1,0.999560,0.999560,0.000360,0.000353,0.000087,-0.000817,0.000360,0.000353,0.000087,-0.000817,0.000360,0.000353,0.000087,-0.000817
1,0.119335,0.001,4.035293e-07,3.971887e-07,0.001000,-2.032879e-20,-9.339797e-11,-6.487907e-10,4.785573e-07,2.114960e-07,4.467975e-10,2.264564e-10,3.137436e-07,4.377546e-08,-1.865087e-10,1.421533e-10,-1.666648e-07,2.663718e-08,-5.465652e-10,-3.272127e-10,-4.611297e-07,-3.620670e-08,-1.118692e-11,-2.238098e-10,-5.510873e-08,-1.038334e-08,2.791711e-10,2.311489e-10,2.329712e-07,1.051521e-07,7.558033e-11,2.107923e-10,4.213944e-08,-1.177404e-08,-2.730901e-10,-2.024192e-10,-1.532516e-07,-5.505051e-08,9.660679e-11,-1.194197e-10,1.403687e-07,-2.428563e-08,1mV,1,2,1.581362,4.035293e-07,3.881329e-07,-9.339797e-11,-6.487907e-10,0.000404,0.000388,-0.000093,-0.000649,0.000404,0.000397,-0.000093,-0.000649,0.566211,0.000655,0.000501,0.016791,0.000003,1,0.999905,0.999905,1,0.999558,0.999558,0.000404,0.000397,-0.000093,-0.000649,0.000404,0.000397,-0.000093,-0.000649,0.000404,0.000397,-0.000093,-0.000649
2,0.150240,0.0009999,4.523700e-07,4.460860e-07,0.001000,9.486769e-20,-1.935061e-10,-3.516726e-10,2.733893e-07,4.409188e-08,5.004613e-10,4.626549e-10,4.484382e-07,-7.147300e-08,-8.007073e-11,8.420214e-11,9.365113e-08,-1.839536e-08,-5.409837e-10,-3.938363e-10,-3.415950e-07,5.135947e-08,-7.067951e-11,-1.968812e-10,3.564944e-08,-1.417029e-07,2.996963e-10,3.998319e-11,9.160415e-08,-1.148292e-07,4.849020e-11,2.310708e-10,4.221248e-09,1.903346e-08,-2.188515e-10,-1.487758e-10,-1.189714e-07,1.105995e-08,-5.979940e-11,3.879897e-11,-2.500749e-08,1.423364e-07,1mV,1,3,1.589897,4.523700e-07,4.346854e-07,-1.935061e-10,-3.516726e-10,0.000452,0.000435,-0.000194,-0.000352,0.000452,0.000446,-0.000194,-0.000352,0.635320,0.000401,0.000682,0.016791,0.000002,1,0.999905,0.999905,1,0.999558,0.999558,0.000452,0.000446,-0.000194,-0.000352,0.000452,0.000446,-0.000194,-0.000352,0.000452,0.000446,-0.000194,-0.000352
3,0.189012,0.001,5.059514e-07,5.007543e-07,0.001000,-2.032879e-20,6.220624e-11,-5.464256e-10,2.752424e-07,-1.908649e-07,5.522691e-10,3.574026e-10,4.747942e-07,3.431016e-07,-3.802952e-10,-3.650356e-11,-2.333305e-07,3.051192e-08,-5.248978e-10,-3.885814e-10,-2.706785e-07,-8.015905e-08,5.126480e-11,-1.614626e-10,-2.730530e-08,-7.748513e-08,2.767869e-10,2.959406e-10,9.182973e-08,1.126101e-07,1.057279e-10,2.010512e-10,9.683833e-08,-3.234306e-08,-2.536576e-10,-3.125619e-10,-1.943091e-07,-5.522583e-08,8.328654e-11,-1.107785e-10,-6.161893e-

In [66]:
save_path = os.path.join(current_file_path,"preprocessed_data_final.csv")
df_12_freq.to_csv(save_path,index=False)

In [67]:
'''
#Initalizations
dfs = {}
keep_cols = ['Freq','Vmod','IHr1','IHi1', 'VHr1', 
 'VHi1', 'IHr2', 'IHi2', 'VHr2', 'VHi2', 
 'IHr3', 'IHi3','VHr3', 'VHi3', 'IHr4', 
 'IHi4', 'VHr4', 'VHi4', 'IHr5', 'IHi5', 
 'VHr5','VHi5', 'IHr6', 'IHi6', 'VHr6', 
 'VHi6', 'IHr7', 'IHi7', 'VHr7', 'VHi7',
 'IHr8', 'IHi8', 'VHr8', 'VHi8', 'IHr9', 
 'IHi9', 'VHr9', 'VHi9', 'IHr10','IHi10', 
 'VHr10', 'VHi10']


# Iterate over files in the directory
for filename in os.listdir(folder_path):
    if filename.endswith('.DTA') and filename.startswith("EISPOT"):
        file_path = os.path.join(folder_path, filename)
        
        # Open and read the file content, skipping the first 20 lines
        with open(file_path, "r", encoding="ISO-8859-1") as f:
            # Skip the first 57 lines
            for _ in range(57):
                next(f)
                
            # Read the remaining lines and create a DataFrame
            content = f.readlines()
            df = pd.DataFrame([line.split('\t') for line in content if line.strip()])
            df = df.map(lambda x: x.strip() if isinstance(x, str) else x) # Remove newline characters from the last column
            df = df.drop(0, axis=1) #drops column numbers
            df.columns = df.iloc[0] # Set the header names using the values from the first row
            df = df.drop(0) # Drop the first row (since it's now used as header)
            df = df.iloc[1:] # Drops row with units
            df = df.dropna(how='all')
            df = df.reset_index(drop=True)
            df = df[keep_cols]
            
            #Saving Files to Output Folder
            filename_stripped = filename[:-4]
            dfs[filename_stripped] = df
dfs['EISPOT_fcn_10mV_2']


'''

'\n#Initalizations\ndfs = {}\nkeep_cols = [\'Freq\',\'Vmod\',\'IHr1\',\'IHi1\', \'VHr1\', \n \'VHi1\', \'IHr2\', \'IHi2\', \'VHr2\', \'VHi2\', \n \'IHr3\', \'IHi3\',\'VHr3\', \'VHi3\', \'IHr4\', \n \'IHi4\', \'VHr4\', \'VHi4\', \'IHr5\', \'IHi5\', \n \'VHr5\',\'VHi5\', \'IHr6\', \'IHi6\', \'VHr6\', \n \'VHi6\', \'IHr7\', \'IHi7\', \'VHr7\', \'VHi7\',\n \'IHr8\', \'IHi8\', \'VHr8\', \'VHi8\', \'IHr9\', \n \'IHi9\', \'VHr9\', \'VHi9\', \'IHr10\',\'IHi10\', \n \'VHr10\', \'VHi10\']\n\n\n# Iterate over files in the directory\nfor filename in os.listdir(folder_path):\n    if filename.endswith(\'.DTA\') and filename.startswith("EISPOT"):\n        file_path = os.path.join(folder_path, filename)\n\n        # Open and read the file content, skipping the first 20 lines\n        with open(file_path, "r", encoding="ISO-8859-1") as f:\n            # Skip the first 57 lines\n            for _ in range(57):\n                next(f)\n\n            # Read the remaining lines and create a DataFrame\n   